In [2]:
import pandas as pd

path = r"D:\ResistMap-AI\data\raw\klebsiella_ast.tsv"
ast = pd.read_csv(path, sep="\t", dtype="string")

print("Rows and columns:", ast.shape)
print("Columns:", ast.columns.tolist())
display(ast.head())

Rows and columns: (32326, 19)
Columns: ['#BioSample', 'Organism group', 'Scientific name', 'Isolation type', 'BioProject', 'Location', 'Isolation source', 'Isolate', 'Antibiotic', 'Resistance phenotype', 'Collection date', 'Measurement sign', 'MIC (mg/L)', 'Disk diffusion (mm)', 'Laboratory typing platform', 'Vendor', 'Laboratory typing method version or reagent', 'Testing standard', 'Create date']


,#BioSample,Organism group,Scientific name,Isolation type,BioProject,Location,Isolation source,Isolate,Antibiotic,Resistance phenotype,Collection date,Measurement sign,MIC (mg/L),Disk diffusion (mm),Laboratory typing platform,Vendor,Laboratory typing method version or reagent,Testing standard,Create date
0,SAMN02141969,Klebsiella pneumoniae,Klebsiella pneumoniae UHKPC28,<NA>,PRJNA187281,<NA>,<NA>,PDT000015887.2,amikacin,intermediate,<NA>,==,32.0,<NA>,Sensititre,TREK Diagnostic Systems,<NA>,CLSI,2013-07-01T19:02:21Z
1,SAMN02142017,Klebsiella pneumoniae,Klebsiella pneumoniae UHKPC47,<NA>,PRJNA187283,<NA>,<NA>,PDT000015889.2,amikacin,intermediate,<NA>,==,32.0,<NA>,Sensititre,TREK Diagnostic Systems,<NA>,CLSI,2013-07-01T19:02:37Z
2,SAMN02141978,Klebsiella pneumoniae,Klebsiella pneumoniae UHKPC23,<NA>,PRJNA187280,<NA>,<NA>,PDT000015890.2,amikacin,intermediate,<NA>,==,32.0,<NA>,Sensititre,TREK Diagnostic Systems,<NA>,CLSI,2013-05-23T13:06:26Z
3,SAMN02142026,Klebsiella pneumoniae,Klebsiella pneumoniae UHKPC69,<NA>,PRJNA187285,<NA>,<NA>,PDT000015891.2,amikacin,susceptible,<NA>,==,16.0,<NA>,Sensititre,TREK Diagnostic Systems,<NA>,CLSI,2013-07-01T19:10:57Z
4,SAMN02142027,Klebsiella pneumoniae,Klebsiella pneumoniae UHKPC77,<NA>,PRJNA187286,<NA>,<NA>,PDT000015892.2,amikacin,susceptible,<NA>,==,16.0,<NA>,Sensititre,TREK Diagnostic Systems,<NA>,CLSI,2013-07-01T19:11:14Z


In [3]:
from pathlib import Path
import pandas as pd

root = Path(r"D:\ResistMap-AI")

# Read original exports; do not use the 3,000-sample prototype.
ast_audit = pd.read_csv(
    root / "data/raw/klebsiella_ast.tsv",
    sep="\t", dtype="string"
)
genomes = pd.read_csv(
    root / "data/raw/isolates_with_metadata.tsv",
    sep="\t", dtype="string"
)

# NCBI adds '#' to the first column header.
for frame in [ast_audit, genomes]:
    frame.columns = frame.columns.str.lstrip("#").str.strip()
    assert frame.columns.is_unique, "Duplicate column names."
    frame["BioSample"] = frame["BioSample"].str.strip().str.upper()
    frame["BioSample"] = frame["BioSample"].replace("", pd.NA)

required_ast = {"BioSample", "Scientific name",
                "Antibiotic", "Resistance phenotype"}
required_genome = {"BioSample", "AMR genotypes"}

assert required_ast.issubset(ast_audit.columns), "AST columns missing."
assert required_genome.issubset(genomes.columns), "Genome columns missing."

# Keep this species, including names with strain/subspecies suffixes.
species_ok = ast_audit["Scientific name"].str.strip().str.match(
    r"^Klebsiella pneumoniae(?:\s|$)", na=False
)
ast_audit = ast_audit.loc[species_ok].copy()

for column in ["Antibiotic", "Resistance phenotype"]:
    ast_audit[column] = ast_audit[column].str.strip().str.lower()
    ast_audit[column] = ast_audit[column].replace("", pd.NA)

print("AST phenotype values:")
display(
    ast_audit["Resistance phenotype"]
    .value_counts(dropna=False)
)

# Genome matching requires a BioSample and an available gene result.
has_genome = (
    genomes["BioSample"].notna()
    & genomes["AMR genotypes"].fillna("").str.strip().ne("")
)
genome_ids = set(genomes.loc[has_genome, "BioSample"])

print("Unique AST BioSamples:",
      ast_audit["BioSample"].nunique())
print("AST BioSamples matching genomic export:",
      ast_audit.loc[
          ast_audit["BioSample"].isin(genome_ids), "BioSample"
      ].nunique())

# Audit conflicts BEFORE excluding intermediate phenotypes.
known = ast_audit.loc[
    ast_audit["Resistance phenotype"].isin(
        ["susceptible", "intermediate", "resistant"]
    )
].dropna(subset=["BioSample", "Antibiotic"]).copy()

keys = ["BioSample", "Antibiotic"]
label_counts = known.groupby(keys)["Resistance phenotype"].transform("nunique")
conflicting = label_counts.gt(1)

print("Conflicting BioSample–antibiotic pairs:",
      known.loc[conflicting, keys].drop_duplicates().shape[0])

# Initial binary task: clear S/R labels, with no S/I/R conflicts.
eligible = known.loc[
    ~conflicting
    & known["Resistance phenotype"].isin(["susceptible", "resistant"])
    & known["BioSample"].isin(genome_ids)
].drop_duplicates(keys).copy()

counts = (
    eligible.groupby(["Antibiotic", "Resistance phenotype"])["BioSample"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=["susceptible", "resistant"], fill_value=0)
)

counts["total"] = counts["susceptible"] + counts["resistant"]
counts["smaller_class"] = counts[["susceptible", "resistant"]].min(axis=1)
counts["resistant_%"] = (
    100 * counts["resistant"] / counts["total"]
).round(1)

print("\nMatched sample counts by antibiotic:")
display(
    counts.sort_values(
        ["smaller_class", "total"], ascending=False
    ).head(20)
)

AST phenotype values:


Resistance phenotype
susceptible                   13627
resistant                     13563
not defined                    3570
intermediate                   1455
nonsusceptible                   64
susceptible-dose dependent       47
Name: count, dtype: int64[pyarrow]

Unique AST BioSamples: 2401
AST BioSamples matching genomic export: 2146
Conflicting BioSample–antibiotic pairs: 2

Matched sample counts by antibiotic:


Resistance phenotype,susceptible,resistant,total,smaller_class,resistant_%
Antibiotic,,,,,
ceftazidime,943,798,1741,798,45.8
ceftriaxone,656,773,1429,656,54.1
piperacillin-tazobactam,796,581,1377,581,42.2
ciprofloxacin,508,617,1125,508,54.8
meropenem,613,504,1117,504,45.1
gentamicin,496,562,1058,496,53.1
cefepime,521,487,1008,487,48.3
trimethoprim-sulfamethoxazole,482,644,1126,482,57.2
ertapenem,589,418,1007,418,41.5


In [4]:
TARGET = "ceftazidime"

# Eligible labels from the previous audit.
target_labels = eligible.loc[
    eligible["Antibiotic"].eq(TARGET),
    ["BioSample", "Resistance phenotype"]
].copy()

assert target_labels["BioSample"].is_unique

# Inspect original AST records, preserving repeated tests and their metadata.
target_ast = ast_audit.loc[
    ast_audit["Antibiotic"].eq(TARGET)
    & ast_audit["BioSample"].isin(target_labels["BioSample"])
].copy()

target_ast["standard_audit"] = (
    target_ast["Testing standard"]
    .fillna("UNKNOWN")
    .str.strip()
    .str.upper()
    .replace("", "UNKNOWN")
)

print("Target:", TARGET)
print("Eligible labelled samples:", len(target_labels))

print("\nUnique BioSamples by testing standard and phenotype:")
display(
    target_ast.groupby(
        ["standard_audit", "Resistance phenotype"]
    )["BioSample"].nunique().unstack(fill_value=0)
)

print("\nMost frequent BioProjects:")
display(
    target_ast.groupby("BioProject")["BioSample"]
    .nunique().sort_values(ascending=False).head(10)
)

target_genomes = genomes.loc[
    genomes["BioSample"].isin(target_labels["BioSample"])
].copy()

genome_audit = target_genomes.groupby("BioSample").agg(
    genomic_records=("BioSample", "size"),
    distinct_amr_strings=("AMR genotypes", "nunique")
)

print("\nBioSamples with multiple genomic records:",
      int(genome_audit["genomic_records"].gt(1).sum()))

print("BioSamples with differing AMR result strings:",
      int(genome_audit["distinct_amr_strings"].gt(1).sum()))

print("\nGenomic scientific names:")
display(
    target_genomes["Scientific name"].value_counts(dropna=False).head(10)
)

Target: ceftazidime
Eligible labelled samples: 1741

Unique BioSamples by testing standard and phenotype:


Resistance phenotype,resistant,susceptible
standard_audit,,
CLSI,579,270
EUCAST,168,634
UNKNOWN,51,39



Most frequent BioProjects:


BioProject
PRJNA1322038    252
PRJNA777643     204
PRJNA278886     162
PRJNA376414     132
PRJNA288601     129
PRJNA1160997     95
PRJNA835677      92
PRJNA717739      90
PRJNA922900      90
PRJNA396774      74
Name: BioSample, dtype: int64


BioSamples with multiple genomic records: 0
BioSamples with differing AMR result strings: 0

Genomic scientific names:


Scientific name
Klebsiella pneumoniae    1741
Name: count, dtype: int64[pyarrow]

In [5]:
# Primary cohort: explicit CLSI results for ceftazidime.
clsi_ids = target_ast.loc[
    target_ast["standard_audit"].eq("CLSI"),
    "BioSample"
].dropna().unique()

# Preserve project information from every relevant AST record.
def collect_projects(values):
    projects = set()
    for value in values.dropna():
        # Handles records containing multiple project accessions.
        projects.update(re.findall(r"PRJ[A-Z]+\d+", str(value).upper()))
    return "|".join(sorted(projects))

import re

project_map = (
    target_ast.groupby("BioSample")["BioProject"]
    .agg(collect_projects)
    .rename("project_accessions")
    .reset_index()
)

genomic_columns = [
    "BioSample", "Isolate", "Scientific name",
    "AMR genotypes", "Collection date", "Location"
]
if "SNP cluster" in target_genomes.columns:
    genomic_columns.append("SNP cluster")

supervised = (
    target_labels.loc[target_labels["BioSample"].isin(clsi_ids)]
    .merge(
        target_genomes[genomic_columns],
        on="BioSample",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        project_map,
        on="BioSample",
        how="left",
        validate="one_to_one"
    )
    .reset_index(drop=True)
)

supervised["target"] = (
    supervised["Resistance phenotype"]
    .map({"susceptible": 0, "resistant": 1})
)

assert supervised["target"].notna().all()
assert supervised["BioSample"].is_unique

print("CLSI cohort rows:", len(supervised))
print("\nClass counts:")
print(supervised["Resistance phenotype"].value_counts())

projects = supervised["project_accessions"].fillna("")
print("\nMissing BioProject:", int(projects.eq("").sum()))
print("Multiple BioProjects:",
      int(projects.str.contains("|", regex=False).sum()))

print("\nLargest project groups:")
display(
    pd.crosstab(
        projects.replace("", "UNKNOWN"),
        supervised["Resistance phenotype"]
    )
    .assign(total=lambda t: t.sum(axis=1))
    .sort_values("total", ascending=False)
    .head(15)
)

if "SNP cluster" in supervised.columns:
    print("\nSNP cluster examples:")
    display(supervised["SNP cluster"].value_counts(dropna=False).head(10))

CLSI cohort rows: 849

Class counts:
Resistance phenotype
resistant      579
susceptible    270
Name: count, dtype: int64[pyarrow]

Missing BioProject: 0
Multiple BioProjects: 0

Largest project groups:


Resistance phenotype,resistant,susceptible,total
project_accessions,,,
PRJNA278886,129,33,162
PRJNA288601,125,4,129
PRJNA376414,105,20,125
PRJNA922900,0,87,87
PRJNA396774,6,66,72
PRJNA308116,32,7,39
PRJNA1156689,31,5,36
PRJNA292902,21,1,22
PRJNA292904,21,0,21



SNP cluster examples:


SNP cluster
<NA>                318
PDS000074884.28      33
PDS000148755.30      30
PDS000085658.24      18
PDS000133600.3        7
PDS000041740.213      7
PDS000085647.10       6
PDS000141295.31       6
PDS000085584.39       6
PDS000006932.31       6
Name: count, dtype: int64[pyarrow]

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

cohort = supervised.reset_index(drop=True).copy()

# Genomic inputs only. Preserve complete genes and point mutations.
def parse_features(value):
    if pd.isna(value) or not str(value).strip():
        raise ValueError("Missing AMR results.")

    features = {}
    for token in str(value).split(","):
        token = token.strip()
        if not token:
            continue
        if "=" not in token:
            raise ValueError(f"Unexpected AMR entry: {token!r}")

        name, status = token.rsplit("=", 1)
        name, status = name.strip(), status.strip().upper()

        if status == "COMPLETE":
            features[f"gene:{name}"] = 1
        elif status == "POINT":
            features[f"mutation:{name}"] = 1

    return features

feature_dicts = cohort["AMR genotypes"].map(parse_features).tolist()
y = cohort["target"].to_numpy(dtype=int)

# Join samples connected by any of the grouping rules.
parent = list(range(len(cohort)))

def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def join(i, j):
    parent[find(i)] = find(j)

seen = {}
for i, row in cohort.iterrows():
    keys = []

    for project in str(row["project_accessions"]).split("|"):
        if project.strip():
            keys.append(("project", project.strip()))

    snp = row.get("SNP cluster", pd.NA)
    if pd.notna(snp) and str(snp).strip().startswith("PDS"):
        # Ignore accession version when grouping.
        keys.append(("snp", str(snp).strip().split(".")[0]))

    keys.append(("profile", tuple(sorted(feature_dicts[i]))))

    for key in keys:
        if key in seen:
            join(i, seen[key])
        else:
            seen[key] = i

groups = np.array([find(i) for i in range(len(cohort))])

group_sizes = pd.Series(groups).value_counts()
print("Connected groups:", len(group_sizes))
print("Largest group:", int(group_sizes.max()), "samples")

if len(group_sizes) < 5:
    raise ValueError(
        "Too few independent groups. Inspect grouping before splitting."
    )

# Fixed split: 20% of groups, not necessarily 20% of samples.
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
train_idx, test_idx = next(
    splitter.split(cohort, y, groups=groups)
)

assert not set(groups[train_idx]) & set(groups[test_idx])

for name, idx in [("TRAIN", train_idx), ("TEST", test_idx)]:
    counts = np.bincount(y[idx], minlength=2)
    print(
        f"{name}: {len(idx)} samples | "
        f"susceptible={counts[0]} | resistant={counts[1]}"
    )
    if (counts == 0).any():
        raise ValueError(
            f"{name} contains only one class. "
            "This split cannot support the planned evaluation."
        )

# Save the split so future experiments use the same partition.
manifest = cohort[["BioSample", "Isolate"]].copy()
manifest["group"] = groups
manifest["partition"] = "train"
manifest.loc[test_idx, "partition"] = "test"

split_path = root / "results/ceftazidime_clsi_split.csv"
split_path.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(split_path, index=False)

print("Group overlap: 0")
print("Saved:", split_path)

Connected groups: 7
Largest group: 823 samples
TRAIN: 10 samples | susceptible=2 | resistant=8
TEST: 839 samples | susceptible=268 | resistant=571
Group overlap: 0
Saved: D:\ResistMap-AI\results\ceftazidime_clsi_split.csv


In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

project_groups = cohort["project_accessions"].fillna("").str.strip()
assert project_groups.ne("").all()
assert not project_groups.str.contains("|", regex=False).any()

profile_keys = [
    tuple(sorted(features.items()))
    for features in feature_dicts
]

def snp_key(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text.split(".")[0] if text.startswith("PDS") else None

snp_keys = cohort["SNP cluster"].map(snp_key).tolist()

# Choose the first split meeting predefined size/class requirements.
# No model predictions or performance scores are used for selection.
splitter = GroupShuffleSplit(
    n_splits=100, test_size=0.20, random_state=42
)

chosen = None

for candidate_train, candidate_test in splitter.split(
    cohort, y, groups=project_groups
):
    test_profiles = {profile_keys[i] for i in candidate_test}
    test_snps = {
        snp_keys[i] for i in candidate_test
        if snp_keys[i] is not None
    }

    # Purge training samples directly overlapping the test set.
    retained_train = np.array([
        i for i in candidate_train
        if profile_keys[i] not in test_profiles
        and (snp_keys[i] is None or snp_keys[i] not in test_snps)
    ], dtype=int)

    train_counts = np.bincount(y[retained_train], minlength=2)
    test_counts = np.bincount(y[candidate_test], minlength=2)

    if (
        len(retained_train) >= 400
        and 100 <= len(candidate_test) <= 250
        and train_counts.min() >= 50
        and test_counts.min() >= 20
        and project_groups.iloc[retained_train].nunique() >= 5
    ):
        chosen = (
            retained_train,
            candidate_test,
            np.setdiff1d(candidate_train, retained_train)
        )
        break

if chosen is None:
    raise ValueError(
        "No split meets the predefined requirements. "
        "Stop here: the cohort or evaluation design needs review."
    )

train_idx, test_idx, excluded_idx = chosen
groups = project_groups.to_numpy()

# Verify the three overlap controls.
assert not set(groups[train_idx]) & set(groups[test_idx])
assert not (
    {profile_keys[i] for i in train_idx}
    & {profile_keys[i] for i in test_idx}
)
assert not (
    {snp_keys[i] for i in train_idx if snp_keys[i] is not None}
    & {snp_keys[i] for i in test_idx if snp_keys[i] is not None}
)

for name, idx in [("TRAIN", train_idx), ("TEST", test_idx)]:
    counts = np.bincount(y[idx], minlength=2)
    print(
        f"{name}: {len(idx)} samples | "
        f"susceptible={counts[0]} | resistant={counts[1]} | "
        f"projects={len(set(groups[idx]))}"
    )

print("Training samples excluded for overlap:", len(excluded_idx))
print("Project / identical-profile / known-SNP overlap: 0")

manifest_v2 = cohort[["BioSample", "Isolate"]].copy()
manifest_v2["BioProject"] = groups
manifest_v2["partition"] = "excluded_overlap"
manifest_v2.loc[train_idx, "partition"] = "train"
manifest_v2.loc[test_idx, "partition"] = "test"

split_path_v2 = root / "results/ceftazidime_clsi_split_v2.csv"
manifest_v2.to_csv(split_path_v2, index=False)
print("Saved:", split_path_v2)

TRAIN: 445 samples | susceptible=83 | resistant=362 | projects=21
TEST: 114 samples | susceptible=41 | resistant=73 | projects=7
Training samples excluded for overlap: 290
Project / identical-profile / known-SNP overlap: 0
Saved: D:\ResistMap-AI\results\ceftazidime_clsi_split_v2.csv


In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Use only the training portion of the saved outer split.
development_idx = np.asarray(train_idx, dtype=int)
assert set(development_idx).isdisjoint(set(test_idx))

cv = StratifiedGroupKFold(
    n_splits=3, shuffle=True, random_state=42
)

cv_folds = []
fold_audit = []

for fold, (fit_local, val_local) in enumerate(
    cv.split(
        np.zeros(len(development_idx)),
        y[development_idx],
        groups[development_idx]
    ),
    start=1
):
    fit_candidates = development_idx[fit_local]
    val_idx = development_idx[val_local]

    val_profiles = {profile_keys[i] for i in val_idx}
    val_snps = {
        snp_keys[i] for i in val_idx
        if snp_keys[i] is not None
    }

    fit_idx = np.array([
        i for i in fit_candidates
        if profile_keys[i] not in val_profiles
        and (snp_keys[i] is None or snp_keys[i] not in val_snps)
    ], dtype=int)

    assert not set(groups[fit_idx]) & set(groups[val_idx])

    fit_counts = np.bincount(y[fit_idx], minlength=2)
    val_counts = np.bincount(y[val_idx], minlength=2)

    fold_audit.append({
        "fold": fold,
        "fit_S": fit_counts[0],
        "fit_R": fit_counts[1],
        "validation_S": val_counts[0],
        "validation_R": val_counts[1],
        "excluded_overlap": len(fit_candidates) - len(fit_idx)
    })

    cv_folds.append((fit_idx, val_idx))

fold_audit = pd.DataFrame(fold_audit)
display(fold_audit)

count_columns = ["fit_S", "fit_R", "validation_S", "validation_R"]
if fold_audit[count_columns].eq(0).any().any():
    raise ValueError(
        "A fold has only one class. Stop here and share the fold table."
    )

print("Validation folds ready. Final-test samples remain separate.")

,fold,fit_S,fit_R,validation_S,validation_R,excluded_overlap
0,1,45,231,38,119,12
1,2,62,216,19,123,25
2,3,55,181,26,120,63


Validation folds ready. Final-test samples remain separate.


In [10]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Stop if Cell 1 did not produce usable folds.
assert not fold_audit[count_columns].eq(0).any().any()

candidates = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),

    "Logistic regression": LogisticRegression(
        C=1.0,
        class_weight="balanced",
        solver="liblinear",
        max_iter=3000,
        random_state=42
    ),

    "Random forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Extra trees": ExtraTreesClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

cv_rows = []

for name, estimator in candidates.items():
    for fold, (fit_idx, val_idx) in enumerate(cv_folds, start=1):
        pipeline = Pipeline([
            ("features", DictVectorizer(sparse=False, dtype=np.float64)),
            ("classifier", clone(estimator))
        ])

        fit_features = [feature_dicts[i] for i in fit_idx]
        val_features = [feature_dicts[i] for i in val_idx]

        pipeline.fit(fit_features, y[fit_idx])

        predictions = pipeline.predict(val_features)
        classes = pipeline.named_steps["classifier"].classes_
        resistant_column = list(classes).index(1)
        probabilities = pipeline.predict_proba(
            val_features
        )[:, resistant_column]

        cv_rows.append({
            "model": name,
            "fold": fold,
            "accuracy": accuracy_score(y[val_idx], predictions),
            "balanced_accuracy": balanced_accuracy_score(
                y[val_idx], predictions
            ),
            "resistant_recall": recall_score(
                y[val_idx], predictions, pos_label=1, zero_division=0
            ),
            "susceptible_recall": recall_score(
                y[val_idx], predictions, pos_label=0, zero_division=0
            ),
            "macro_f1": f1_score(
                y[val_idx], predictions,
                average="macro", zero_division=0
            ),
            "roc_auc": roc_auc_score(y[val_idx], probabilities)
        })

    print(f"Completed: {name}")

cv_results = pd.DataFrame(cv_rows)

metrics = [
    "accuracy", "balanced_accuracy",
    "resistant_recall", "susceptible_recall",
    "macro_f1", "roc_auc"
]

cv_summary = cv_results.groupby("model")[metrics].mean()
cv_summary["balanced_accuracy_sd"] = (
    cv_results.groupby("model")["balanced_accuracy"].std()
)
cv_summary = cv_summary.sort_values(
    "balanced_accuracy", ascending=False
)

print("\nMean validation scores (%) and fold variation:")
display((cv_summary * 100).round(2))

print("\nBalanced accuracy by fold (%):")
display(
    (cv_results.pivot(
        index="model", columns="fold", values="balanced_accuracy"
    ) * 100).round(2)
)

cv_results.to_csv(
    root / "results/ceftazidime_grouped_cv.csv", index=False
)

Completed: Majority baseline
Completed: Logistic regression
Completed: Random forest
Completed: Extra trees

Mean validation scores (%) and fold variation:


,accuracy,balanced_accuracy,resistant_recall,susceptible_recall,macro_f1,roc_auc,balanced_accuracy_sd
model,,,,,,,
Random forest,84.37,78.56,87.68,69.43,74.77,85.47,16.29
Extra trees,85.06,78.47,88.80,68.15,74.99,85.69,16.41
Logistic regression,82.56,77.23,85.44,69.03,72.30,86.31,12.39
Majority baseline,81.54,50.00,100.00,0.00,44.88,50.00,0.00



Balanced accuracy by fold (%):


fold,1,2,3
model,,,
Extra trees,95.27,77.69,62.47
Logistic regression,88.07,79.91,63.72
Majority baseline,50.00,50.00,50.00
Random forest,95.27,77.69,62.72


In [11]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction import DictVectorizer

audit_parts = []
audit_y = np.asarray(y, dtype=int)
development_rows = set(np.asarray(train_idx, dtype=int))
final_test_rows = set(np.asarray(test_idx, dtype=int))

for fold, (fit_rows, val_rows) in enumerate(cv_folds, start=1):
    fit_rows = np.asarray(fit_rows, dtype=int)
    val_rows = np.asarray(val_rows, dtype=int)

    assert set(fit_rows).issubset(development_rows)
    assert set(val_rows).issubset(development_rows)
    assert set(fit_rows).isdisjoint(set(val_rows))
    assert (set(fit_rows) | set(val_rows)).isdisjoint(final_test_rows)

    audit_model = Pipeline([
        ("features", DictVectorizer(sparse=False, dtype=np.float64)),
        ("classifier", clone(candidates["Random forest"]))
    ])

    audit_model.fit(
        [feature_dicts[i] for i in fit_rows],
        audit_y[fit_rows]
    )

    validation_features = [feature_dicts[i] for i in val_rows]
    predicted = audit_model.predict(validation_features)

    classes = audit_model.named_steps["classifier"].classes_
    r_column = list(classes).index(1)
    probability_r = audit_model.predict_proba(
        validation_features
    )[:, r_column]

    part = cohort.iloc[val_rows][
        ["BioSample", "Isolate", "project_accessions"]
    ].copy().reset_index(drop=True)

    part = part.rename(columns={"project_accessions": "BioProject"})
    part["row_id"] = val_rows
    part["fold"] = fold
    part["actual"] = audit_y[val_rows]      # 0 = S, 1 = R
    part["predicted"] = predicted
    part["probability_R"] = probability_r
    part["error"] = part["actual"].ne(part["predicted"])

    audit_parts.append(part)
    print(f"Audited fold {fold}")

oof_audit = pd.concat(audit_parts, ignore_index=True)

# Each development sample must appear in validation exactly once.
assert oof_audit["row_id"].is_unique
assert set(oof_audit["row_id"]) == development_rows

def summarize_errors(frame):
    actual = frame["actual"].to_numpy()
    predicted = frame["predicted"].to_numpy()

    is_s = actual == 0
    is_r = actual == 1

    s_recall = (
        100 * np.mean(predicted[is_s] == 0) if is_s.any() else np.nan
    )
    r_recall = (
        100 * np.mean(predicted[is_r] == 1) if is_r.any() else np.nan
    )

    return {
        "samples": len(frame),
        "S": int(is_s.sum()),
        "R": int(is_r.sum()),
        "errors": int((actual != predicted).sum()),
        "S_as_R": int((is_s & (predicted == 1)).sum()),
        "R_as_S": int((is_r & (predicted == 0)).sum()),
        "accuracy_%": 100 * np.mean(actual == predicted),
        "S_recall_%": s_recall,
        "R_recall_%": r_recall,
        # Leave undefined when a project has only one actual class.
        "balanced_accuracy_%": (s_recall + r_recall) / 2
    }

fold_errors = pd.DataFrame([
    {"fold": fold, **summarize_errors(frame)}
    for fold, frame in oof_audit.groupby("fold")
])

project_errors = pd.DataFrame([
    {"fold": fold, "BioProject": project, **summarize_errors(frame)}
    for (fold, project), frame in oof_audit.groupby(
        ["fold", "BioProject"], dropna=False
    )
]).sort_values(["fold", "errors"], ascending=[True, False])

print("\nErrors by fold:")
display(fold_errors.round(2))

print("\nErrors by BioProject:")
display(project_errors.round(2))

results_dir = root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

oof_audit.to_csv(
    results_dir / "ceftazidime_rf_oof_predictions.csv", index=False
)
project_errors.to_csv(
    results_dir / "ceftazidime_rf_project_errors.csv", index=False
)

Audited fold 1
Audited fold 2
Audited fold 3

Errors by fold:


,fold,samples,S,R,errors,S_as_R,R_as_S,accuracy_%,S_recall_%,R_recall_%,balanced_accuracy_%
0,1,157,38,119,7,2,5,95.54,94.74,95.80,95.27
1,2,142,19,123,33,4,29,76.76,78.95,76.42,77.69
2,3,146,26,120,28,17,11,80.82,34.62,90.83,62.72



Errors by BioProject:


,fold,BioProject,samples,S,R,errors,S_as_R,R_as_S,accuracy_%,S_recall_%,R_recall_%,balanced_accuracy_%
0,1,PRJNA288601,87,2,85,5,2,3,94.25,0.00,96.47,48.24
1,1,PRJNA292904,16,0,16,1,0,1,93.75,NaN,93.75,NaN
6,1,PRJNA861833,2,0,2,1,0,1,50.00,NaN,50.00,NaN
2,1,PRJNA312972,5,0,5,0,0,0,100.00,NaN,100.00,NaN
3,1,PRJNA316321,5,0,5,0,0,0,100.00,NaN,100.00,NaN
4,1,PRJNA523425,2,0,2,0,0,0,100.00,NaN,100.00,NaN
5,1,PRJNA523732,4,0,4,0,0,0,100.00,NaN,100.00,NaN
7,1,PRJNA922900,36,36,0,0,0,0,100.00,100.00,NaN,NaN
8,2,PRJNA376414,101,10,91,29,3,26,71.29,70.00,71.43,70.71
9,2,PRJNA396774,7,5,2,2,1,1,71.43,80.00,50.00,65.00


In [12]:
import numpy as np
import pandas as pd

dev_rows = np.sort(np.asarray(train_idx, dtype=int))
assert set(dev_rows).isdisjoint(set(test_idx))

profile_audit = cohort.iloc[dev_rows][
    ["BioSample", "Isolate", "project_accessions"]
].copy().reset_index(drop=True)

profile_audit = profile_audit.rename(
    columns={"project_accessions": "BioProject"}
)
profile_audit["row_id"] = dev_rows
profile_audit["target"] = np.asarray(y, dtype=int)[dev_rows]

# Compare exactly the feature dictionaries supplied to the models.
# Series avoids the earlier pandas factorize(list) error.
feature_keys = pd.Series(
    [tuple(sorted(feature_dicts[i].items())) for i in dev_rows],
    dtype=object
)
profile_audit["profile_id"] = pd.factorize(
    feature_keys, sort=False
)[0]

profile_counts = pd.crosstab(
    profile_audit["profile_id"],
    profile_audit["target"]
).reindex(columns=[0, 1], fill_value=0)

profile_counts.columns = ["S", "R"]
profile_counts["samples"] = profile_counts[["S", "R"]].sum(axis=1)
profile_counts["projects"] = (
    profile_audit.groupby("profile_id")["BioProject"].nunique()
)
profile_counts["minimum_errors"] = (
    profile_counts[["S", "R"]].min(axis=1)
)

mixed_profiles = profile_counts.loc[
    (profile_counts["S"] > 0) & (profile_counts["R"] > 0)
].sort_values("minimum_errors", ascending=False)

minimum_errors = int(profile_counts["minimum_errors"].sum())
max_fit_accuracy = 100 * (1 - minimum_errors / len(dev_rows))

print("Development samples:", len(dev_rows))
print("Distinct feature profiles:", len(profile_counts))
print("Profiles containing both S and R:", len(mixed_profiles))
print("Samples in those profiles:", int(mixed_profiles["samples"].sum()))
print("Minimum errors with identical-input predictions:", minimum_errors)
print(
    "Maximum fit accuracy on these rows using only current features:",
    f"{max_fit_accuracy:.2f}%"
)

print("\nIdentical feature profiles with different labels:")
display(mixed_profiles.head(15))

mixed_samples = profile_audit.loc[
    profile_audit["profile_id"].isin(mixed_profiles.index)
].copy()
mixed_samples["phenotype"] = mixed_samples["target"].map(
    {0: "susceptible", 1: "resistant"}
)

results_dir = root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

profile_counts.to_csv(results_dir / "ceftazidime_profile_label_audit.csv")
mixed_samples.to_csv(
    results_dir / "ceftazidime_mixed_label_samples.csv", index=False
)

Development samples: 445
Distinct feature profiles: 318
Profiles containing both S and R: 4
Samples in those profiles: 30
Minimum errors with identical-input predictions: 5
Maximum fit accuracy on these rows using only current features: 98.88%

Identical feature profiles with different labels:


,S,R,samples,projects,minimum_errors
profile_id,,,,,
92,2,8,10,2,2
0,1,14,15,2,1
1,1,1,2,2,1
43,1,2,3,1,1


In [13]:
import numpy as np
import pandas as pd

raw_review = mixed_samples.copy().reset_index(drop=True)
review_rows = raw_review["row_id"].to_numpy(dtype=int)

assert set(review_rows).issubset(set(train_idx))
assert set(review_rows).isdisjoint(set(test_idx))

raw_review["AMR genotypes"] = (
    cohort.iloc[review_rows]["AMR genotypes"].to_numpy()
)

def full_amr_key(value):
    if pd.isna(value):
        raise ValueError("Missing AMR report in review samples.")

    entries = []
    for token in str(value).split(","):
        token = token.strip()
        if not token:
            continue
        if "=" not in token:
            raise ValueError(f"Unexpected AMR entry: {token!r}")

        name, status = token.rsplit("=", 1)
        entries.append(f"{name.strip()}={status.strip().upper()}")

    # Ignore entry order; preserve all reported statuses.
    return tuple(sorted(entries))

raw_review["full_report_key"] = (
    raw_review["AMR genotypes"].map(full_amr_key)
)
raw_review["full_report_id"] = pd.factorize(
    raw_review["full_report_key"], sort=False
)[0]

full_counts = pd.crosstab(
    raw_review["full_report_id"], raw_review["target"]
).reindex(columns=[0, 1], fill_value=0)

comparison_rows = []

for profile_id, frame in raw_review.groupby("profile_id"):
    reports = [set(key) for key in frame["full_report_key"]]
    varying_entries = sorted(
        set.union(*reports) - set.intersection(*reports)
    )

    report_ids = frame["full_report_id"].unique()
    remaining_errors = int(
        full_counts.loc[report_ids, [0, 1]].min(axis=1).sum()
    )

    comparison_rows.append({
        "profile_id": profile_id,
        "samples": len(frame),
        "S": int(frame["target"].eq(0).sum()),
        "R": int(frame["target"].eq(1).sum()),
        "full_report_variants": len(report_ids),
        "remaining_minimum_errors": remaining_errors,
        "varying_entries": "; ".join(varying_entries) or "(none)"
    })

raw_comparison = pd.DataFrame(comparison_rows)

with pd.option_context("display.max_colwidth", 150):
    display(raw_comparison)

results_dir = root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

raw_comparison.to_csv(
    results_dir / "ceftazidime_raw_report_comparison.csv",
    index=False
)
raw_review.drop(columns="full_report_key").to_csv(
    results_dir / "ceftazidime_raw_report_review_samples.csv",
    index=False
)

,profile_id,samples,S,R,full_report_variants,remaining_minimum_errors,varying_entries
0,0,15,1,14,6,1,aadA2=PARTIAL_END_OF_CONTIG; blaSHV=PARTIAL_END_OF_CONTIG; mrx(A)=PARTIAL_END_OF_CONTIG
1,1,2,1,1,1,1,(none)
2,43,3,1,2,3,0,aac(6')-Ib=PARTIAL; aac(6')-Ib=PARTIAL_END_OF_CONTIG; blaSHV=PARTIAL_END_OF_CONTIG; oqxA=PARTIAL_END_OF_CONTIG
3,92,10,2,8,3,2,blaOXA=PARTIAL_END_OF_CONTIG; catB3=PARTIAL; oqxA=PARTIAL_END_OF_CONTIG; oqxB=PARTIAL_END_OF_CONTIG


In [14]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    recall_score, f1_score, roc_auc_score
)

# Add partial statuses to existing COMPLETE/POINT features.
dev_rows = np.asarray(train_idx, dtype=int)
dev_set = set(dev_rows)
test_set = set(np.asarray(test_idx, dtype=int))
assert dev_set.isdisjoint(test_set)

partial_features = {}

for row_id in dev_rows:
    features = dict(feature_dicts[row_id])
    report = cohort.iloc[row_id]["AMR genotypes"]

    if pd.isna(report):
        raise ValueError(f"Missing AMR report at row {row_id}")

    for token in str(report).split(","):
        token = token.strip()
        if not token:
            continue
        if "=" not in token:
            raise ValueError(f"Unexpected entry: {token!r}")

        gene, status = token.rsplit("=", 1)
        gene, status = gene.strip(), status.strip().upper()

        if status == "PARTIAL":
            features[f"partial:{gene}"] = 1
        elif status in {"PARTIAL_END_OF_CONTIG", "PARTIAL_CONTIG_END"}:
            features[f"contig_end:{gene}"] = 1

    partial_features[int(row_id)] = features

print(
    "Development samples with additional features:",
    sum(
        len(partial_features[i]) > len(feature_dicts[i])
        for i in dev_rows
    )
)

model_names = [
    "Logistic regression",
    "Random forest",
    "Extra trees"
]

expanded_rows = []
expanded_predictions = []
labels = np.asarray(y, dtype=int)

for name in model_names:
    for fold, (fit_rows, val_rows) in enumerate(cv_folds, start=1):
        fit_rows = np.asarray(fit_rows, dtype=int)
        val_rows = np.asarray(val_rows, dtype=int)

        assert set(fit_rows).issubset(dev_set)
        assert set(val_rows).issubset(dev_set)
        assert set(fit_rows).isdisjoint(set(val_rows))
        assert (set(fit_rows) | set(val_rows)).isdisjoint(test_set)
        assert set(labels[fit_rows]) == {0, 1}
        assert set(labels[val_rows]) == {0, 1}

        experiment = Pipeline([
            ("features", DictVectorizer(sparse=False, dtype=np.float64)),
            ("classifier", clone(candidates[name]))
        ])

        experiment.fit(
            [partial_features[i] for i in fit_rows],
            labels[fit_rows]
        )

        validation_features = [partial_features[i] for i in val_rows]
        predicted = experiment.predict(validation_features)

        classes = experiment.named_steps["classifier"].classes_
        r_column = list(classes).index(1)
        probability = experiment.predict_proba(
            validation_features
        )[:, r_column]

        actual = labels[val_rows]

        expanded_rows.append({
            "model": name,
            "fold": fold,
            "accuracy": accuracy_score(actual, predicted),
            "balanced_accuracy": balanced_accuracy_score(actual, predicted),
            "resistant_recall": recall_score(
                actual, predicted, pos_label=1, zero_division=0
            ),
            "susceptible_recall": recall_score(
                actual, predicted, pos_label=0, zero_division=0
            ),
            "macro_f1": f1_score(
                actual, predicted, average="macro", zero_division=0
            ),
            "roc_auc": roc_auc_score(actual, probability),
            "training_features": len(
                experiment.named_steps["features"].vocabulary_
            )
        })

        expanded_predictions.append(pd.DataFrame({
            "model": name,
            "fold": fold,
            "row_id": val_rows,
            "actual": actual,
            "predicted": predicted,
            "probability_R": probability
        }))

    print("Completed:", name)

expanded_cv = pd.DataFrame(expanded_rows)
expanded_oof = pd.concat(expanded_predictions, ignore_index=True)

# Compare with the original CV results.
old_mean = cv_results.groupby("model")["balanced_accuracy"].mean()
new_mean = expanded_cv.groupby("model").mean(numeric_only=True)

feature_comparison = pd.DataFrame(index=new_mean.index)
feature_comparison["old_balanced_%"] = old_mean.reindex(new_mean.index) * 100
feature_comparison["new_balanced_%"] = new_mean["balanced_accuracy"] * 100
feature_comparison["change_pp"] = (
    feature_comparison["new_balanced_%"]
    - feature_comparison["old_balanced_%"]
)
feature_comparison["new_accuracy_%"] = new_mean["accuracy"] * 100
feature_comparison["new_S_recall_%"] = new_mean["susceptible_recall"] * 100
feature_comparison["new_R_recall_%"] = new_mean["resistant_recall"] * 100
feature_comparison["worst_fold_balanced_%"] = (
    expanded_cv.groupby("model")["balanced_accuracy"].min() * 100
)

print("\nOriginal versus expanded features:")
display(
    feature_comparison.sort_values(
        "new_balanced_%", ascending=False
    ).round(2)
)

print("\nExpanded features — balanced accuracy by fold (%):")
display(
    (expanded_cv.pivot(
        index="model", columns="fold", values="balanced_accuracy"
    ) * 100).round(2)
)

results_dir = root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

expanded_cv.to_csv(
    results_dir / "ceftazidime_partial_features_cv.csv", index=False
)
expanded_oof.to_csv(
    results_dir / "ceftazidime_partial_features_oof.csv", index=False
)
feature_comparison.to_csv(
    results_dir / "ceftazidime_feature_comparison.csv"
)

Development samples with additional features: 274
Completed: Logistic regression
Completed: Random forest
Completed: Extra trees

Original versus expanded features:


,old_balanced_%,new_balanced_%,change_pp,new_accuracy_%,new_S_recall_%,new_R_recall_%,worst_fold_balanced_%
model,,,,,,,
Extra trees,78.47,78.61,0.14,85.28,68.15,89.07,62.05
Random forest,78.56,77.63,-0.92,83.73,68.15,87.12,60.38
Logistic regression,77.23,76.31,-0.92,81.85,67.75,84.88,59.71



Expanded features — balanced accuracy by fold (%):


fold,1,2,3
model,,,
Extra trees,95.69,78.09,62.05
Logistic regression,88.91,80.32,59.71
Random forest,94.43,78.09,60.38


In [15]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import GridSearchCV, ParameterGrid
from sklearn.metrics import make_scorer, recall_score

# Original COMPLETE/POINT features, development samples only.
tuning_idx = np.asarray(train_idx, dtype=int)

assert len(np.unique(tuning_idx)) == len(tuning_idx)
assert set(tuning_idx).isdisjoint(set(test_idx))

X_dev = [feature_dicts[int(i)] for i in tuning_idx]
y_dev = np.asarray(y, dtype=int)[tuning_idx]

# Convert existing cohort indices into development-set positions.
position = {int(row): pos for pos, row in enumerate(tuning_idx)}
cv_local = []

for fit_global, val_global in cv_folds:
    fit_local = np.array(
        [position[int(i)] for i in fit_global], dtype=int
    )
    val_local = np.array(
        [position[int(i)] for i in val_global], dtype=int
    )

    assert set(fit_local).isdisjoint(set(val_local))
    assert set(y_dev[fit_local]) == {0, 1}
    assert set(y_dev[val_local]) == {0, 1}

    cv_local.append((fit_local, val_local))

validation_positions = np.concatenate([v for _, v in cv_local])
assert np.all(
    np.bincount(validation_positions, minlength=len(tuning_idx)) == 1
)

# Predefined, bounded search; includes the original configurations.
parameter_grid = [{
    "classifier": [clone(candidates["Logistic regression"])],
    "classifier__C": [0.01, 0.1, 1.0, 10.0]
}]

for name in ["Random forest", "Extra trees"]:
    parameter_grid.append({
        "classifier": [clone(candidates[name])],
        "classifier__max_depth": [None, 8],
        "classifier__min_samples_leaf": [1, 2, 5],
        "classifier__max_features": ["sqrt", 0.5]
    })

scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
    "S_recall": make_scorer(
        recall_score, pos_label=0, zero_division=0
    ),
    "R_recall": make_scorer(
        recall_score, pos_label=1, zero_division=0
    ),
    "macro_f1": "f1_macro",
    "roc_auc": "roc_auc"
}

search_pipeline = Pipeline([
    ("features", DictVectorizer(sparse=False, dtype=np.float64)),
    ("classifier", clone(candidates["Logistic regression"]))
])

search = GridSearchCV(
    estimator=search_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit="balanced_accuracy",
    cv=cv_local,
    n_jobs=1,
    error_score="raise",
    return_train_score=False,
    verbose=1
)

print("Configurations:", len(ParameterGrid(parameter_grid)))
search.fit(X_dev, y_dev)

# search.best_estimator_ is refitted on the development samples.
tuned_model = search.best_estimator_
tuning_results = pd.DataFrame(search.cv_results_)

leaderboard = pd.DataFrame({
    "candidate": np.arange(len(tuning_results)),
    "model": [
        type(p["classifier"]).__name__
        for p in tuning_results["params"]
    ]
})

for metric in scoring:
    leaderboard[f"{metric}_%"] = (
        tuning_results[f"mean_test_{metric}"] * 100
    )

fold_columns = [
    f"split{i}_test_balanced_accuracy"
    for i in range(len(cv_local))
]

leaderboard["worst_fold_BA_%"] = (
    tuning_results[fold_columns].min(axis=1) * 100
)
leaderboard["parameters"] = [
    {
        key.replace("classifier__", ""): value
        for key, value in parameters.items()
        if key != "classifier"
    }
    for parameters in tuning_results["params"]
]

leaderboard = leaderboard.sort_values(
    "balanced_accuracy_%", ascending=False, kind="stable"
)

print("\nTop configurations — development CV selection scores:")
display(leaderboard[[
    "candidate", "model", "balanced_accuracy_%",
    "worst_fold_BA_%", "accuracy_%", "S_recall_%", "R_recall_%"
]].head(10).round(2))

winner = tuning_results.iloc[search.best_index_]

print("\nSelected model:", type(
    tuned_model.named_steps["classifier"]
).__name__)

print("Selected parameters:", {
    key.replace("classifier__", ""): value
    for key, value in search.best_params_.items()
    if key != "classifier"
})

print("Selected model balanced accuracy by fold (%):", [
    round(float(winner[column]) * 100, 2)
    for column in fold_columns
])

results_dir = root / "results"
results_dir.mkdir(parents=True, exist_ok=True)
leaderboard.to_csv(
    results_dir / "ceftazidime_tuning_results.csv", index=False
)

Configurations: 28
Fitting 3 folds for each of 28 candidates, totalling 84 fits

Top configurations — development CV selection scores:


,candidate,model,balanced_accuracy_%,worst_fold_BA_%,accuracy_%,S_recall_%,R_recall_%
8,8,RandomForestClassifier,83.09,69.07,88.10,75.51,90.68
14,14,RandomForestClassifier,83.09,69.07,88.10,75.51,90.68
9,9,RandomForestClassifier,81.57,67.40,85.97,74.63,88.51
21,21,ExtraTreesClassifier,81.44,68.24,85.74,74.63,88.24
27,27,ExtraTreesClassifier,81.44,68.24,85.74,74.63,88.24
20,20,ExtraTreesClassifier,81.39,68.65,85.96,74.63,88.16
7,7,RandomForestClassifier,81.36,64.81,87.15,72.94,89.78
15,15,RandomForestClassifier,81.30,67.40,85.50,74.63,87.97
13,13,RandomForestClassifier,81.08,64.81,86.72,72.94,89.22
26,26,ExtraTreesClassifier,80.85,68.24,85.03,74.63,87.07



Selected model: RandomForestClassifier
Selected parameters: {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 2}
Selected model balanced accuracy by fold (%): [96.64, 83.57, 69.07]


In [16]:
import json
import joblib
import sklearn
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix
)

# GridSearchCV already fitted this pipeline on all development samples.
final_model = search.best_estimator_

fit_rows = np.asarray(tuning_idx, dtype=int)
eval_rows = np.asarray(test_idx, dtype=int)

assert set(fit_rows) == set(train_idx)
assert set(fit_rows).isdisjoint(set(eval_rows))
assert len(np.unique(eval_rows)) == len(eval_rows)

# Recheck the original split restrictions.
assert set(groups[fit_rows]).isdisjoint(set(groups[eval_rows]))
assert {profile_keys[i] for i in fit_rows}.isdisjoint(
    {profile_keys[i] for i in eval_rows}
)
assert {
    snp_keys[i] for i in fit_rows if snp_keys[i] is not None
}.isdisjoint({
    snp_keys[i] for i in eval_rows if snp_keys[i] is not None
})

results_dir = root / "results"
models_dir = root / "models"
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# Save the selected pipeline and settings before test evaluation.
model_path = models_dir / "ceftazidime_clsi_rf_v1.joblib"
joblib.dump(final_model, model_path)

frozen_settings = {
    "target": "ceftazidime",
    "cohort_standard": "CLSI",
    "features": {
        "COMPLETE": "gene:<name>",
        "POINT": "mutation:<name>",
        "other_statuses": "excluded"
    },
    "labels": {"0": "susceptible", "1": "resistant"},
    "decision_rule": "classifier.predict; default class decision",
    "classifier_parameters": (
        final_model.named_steps["classifier"].get_params()
    ),
    "development_cv_balanced_accuracy": float(search.best_score_),
    "sklearn_version": sklearn.__version__,
    "training_biosamples": (
        cohort.iloc[fit_rows]["BioSample"].astype(str).tolist()
    ),
    "test_biosamples": (
        cohort.iloc[eval_rows]["BioSample"].astype(str).tolist()
    )
}

with open(
    models_dir / "ceftazidime_clsi_rf_v1_metadata.json",
    "w", encoding="utf-8"
) as file:
    json.dump(frozen_settings, file, indent=2)

# Evaluate the frozen model.
X_test_final = [feature_dicts[int(i)] for i in eval_rows]
y_test_final = np.asarray(y, dtype=int)[eval_rows]
assert set(y_test_final) == {0, 1}

test_prediction = final_model.predict(X_test_final)

classes = final_model.named_steps["classifier"].classes_
r_column = list(classes).index(1)
test_probability_r = final_model.predict_proba(
    X_test_final
)[:, r_column]

test_scores = pd.Series({
    "accuracy": accuracy_score(y_test_final, test_prediction),
    "balanced_accuracy": balanced_accuracy_score(
        y_test_final, test_prediction
    ),
    "susceptible_recall": recall_score(
        y_test_final, test_prediction, pos_label=0, zero_division=0
    ),
    "resistant_recall": recall_score(
        y_test_final, test_prediction, pos_label=1, zero_division=0
    ),
    "macro_f1": f1_score(
        y_test_final, test_prediction, average="macro", zero_division=0
    ),
    "roc_auc": roc_auc_score(y_test_final, test_probability_r)
}, name="score")

test_confusion = pd.DataFrame(
    confusion_matrix(y_test_final, test_prediction, labels=[0, 1]),
    index=["Actual susceptible", "Actual resistant"],
    columns=["Predicted susceptible", "Predicted resistant"]
)

test_report = cohort.iloc[eval_rows][
    ["BioSample", "Isolate", "project_accessions"]
].copy().reset_index(drop=True)

test_report["actual"] = y_test_final
test_report["predicted"] = test_prediction
test_report["probability_R"] = test_probability_r
test_report["error"] = y_test_final != test_prediction

vocabulary = set(final_model.named_steps["features"].vocabulary_)
test_report["unseen_feature_count"] = [
    len(set(features) - vocabulary)
    for features in X_test_final
]

print("Final-test samples:", len(eval_rows))
print("Final-test projects:", test_report["project_accessions"].nunique())
print(
    "Correct predictions:",
    int((y_test_final == test_prediction).sum()),
    "/", len(eval_rows)
)

print("\nReserved-test scores (%):")
display((test_scores * 100).round(2).to_frame("score_%"))

print("\nConfusion matrix — sample counts:")
display(test_confusion)

test_scores.to_csv(
    results_dir / "ceftazidime_clsi_test_scores_v1.csv"
)
test_confusion.to_csv(
    results_dir / "ceftazidime_clsi_test_confusion_v1.csv"
)
test_report.to_csv(
    results_dir / "ceftazidime_clsi_test_predictions_v1.csv",
    index=False
)

print("\nSaved model:", model_path)

Final-test samples: 114
Final-test projects: 7
Correct predictions: 100 / 114

Reserved-test scores (%):


,score_%
accuracy,87.72
balanced_accuracy,89.34
susceptible_recall,95.12
resistant_recall,83.56
macro_f1,87.24
roc_auc,97.53



Confusion matrix — sample counts:


,Predicted susceptible,Predicted resistant
Actual susceptible,39,2
Actual resistant,12,61



Saved model: D:\ResistMap-AI\models\ceftazidime_clsi_rf_v1.joblib


In [18]:
%run -i "D:/ResistMap-AI/retrain_ceftazidime_v2.py"

Training 11 candidates across 3 existing folds.
CV fits: 33; followed by one final development fit.
Threshold objective: highest mean development-CV balanced accuracy.
These are selection scores; fresh independent validation is required.
RF_v1_control: fold 1/3 complete
RF_v1_control: fold 2/3 complete
RF_v1_control: fold 3/3 complete
RF_leaf_1: fold 1/3 complete
RF_leaf_1: fold 2/3 complete
RF_leaf_1: fold 3/3 complete
ExtraTrees_leaf_1: fold 1/3 complete
ExtraTrees_leaf_1: fold 2/3 complete
ExtraTrees_leaf_1: fold 3/3 complete
RF_leaf_2: fold 1/3 complete
RF_leaf_2: fold 2/3 complete
RF_leaf_2: fold 3/3 complete
ExtraTrees_leaf_2: fold 1/3 complete
ExtraTrees_leaf_2: fold 2/3 complete
ExtraTrees_leaf_2: fold 3/3 complete
RF_leaf_4: fold 1/3 complete
RF_leaf_4: fold 2/3 complete
RF_leaf_4: fold 3/3 complete
ExtraTrees_leaf_4: fold 1/3 complete
ExtraTrees_leaf_4: fold 2/3 complete
ExtraTrees_leaf_4: fold 3/3 complete
Logistic_C_0.1: fold 1/3 complete
Logistic_C_0.1: fold 2/3 complete
L

,candidate,threshold,balanced_accuracy_%,accuracy_%,S_recall_%,R_recall_%,worst_fold_BA_%,roc_auc_%
0,RF_leaf_4,0.40,83.66,90.37,73.35,93.98,67.15,87.10
1,RF_v1_control,0.50,83.09,88.10,75.51,90.68,69.07,86.95
2,RF_leaf_2,0.50,82.82,87.63,75.51,90.14,69.07,86.91
5,ExtraTrees_leaf_4,0.45,82.26,87.16,74.63,89.88,69.49,85.84
7,ExtraTrees_leaf_1,0.70,82.08,86.74,75.51,88.66,68.24,84.52
14,RF_leaf_1,0.55,81.54,85.79,75.51,87.58,68.65,85.69
15,ExtraTrees_leaf_2,0.50,81.39,85.96,74.63,88.16,68.65,86.58
29,HistGB_leaves_7,0.50,80.23,86.01,71.19,89.27,65.64,86.55



Selected: RF_leaf_4; threshold: 0.40


,fold,threshold,balanced_accuracy_%,accuracy_%,S_recall_%,R_recall_%,roc_auc_%
0,1,0.4,97.42,97.45,97.37,97.48,99.04
1,2,0.4,86.41,88.03,84.21,88.62,90.59
2,3,0.4,67.15,85.62,38.46,95.83,71.67



Saved classifier with threshold: D:\ResistMap-AI\results\ceftazidime_v2_20261001T202849_616568Z\model_with_threshold.joblib
V2 predict() applies the selected threshold automatically.
AUC and selection scores are not verified 99% accuracy.


In [19]:
print("Available genomic columns:")
print(genomes.columns.tolist())

lookup_cols = [
    c for c in genomes.columns
    if any(term in c.lower() for term in (
        "biosample", "isolate", "assembly",
        "genbank", "refseq", "sra"
    ))
]

development_ids = cohort.iloc[train_idx]["BioSample"]

sequence_lookup = genomes.loc[
    genomes["BioSample"].isin(development_ids),
    lookup_cols
].copy()

print("\nMatched development genomic records:", len(sequence_lookup))
display(sequence_lookup.head(10))

Available genomic columns:
['Organism group', 'Strain', 'Isolate identifiers', 'Serovar', 'Collection date', 'Scientific name', 'Isolate', 'Create date', 'Location', 'Isolation source', 'Isolation type', 'Food origin', 'SNP cluster', 'Min-same', 'Min-diff', 'BioSample', 'Assembly', 'AMR genotypes', 'Computed types']

Matched development genomic records: 445


,Isolate identifiers,Isolate,BioSample,Assembly
302,"0000-0053-2559,""66"",""SRS1016976""",PDT000077408.3,SAMN03892126,GCA_022316805.1
303,"0000-0051-8445,""24"",""SRS1016971""",PDT000077413.3,SAMN03892119,GCA_022316865.1
304,"0000-0051-2251,""36"",""SRS1016970""",PDT000077414.3,SAMN03892120,GCA_022316825.1
311,"0000-0054-1472,""104"",""SRS1073968""",PDT000084309.3,SAMN04096282,GCA_022316565.1
400,"0000-0056-4336,""178"",""SRS1296145""",PDT000107928.2,SAMN04456590,GCA_022316405.1
403,"MRSN 12232,""MRSN12232"",""SRS1308337""",PDT000111516.2,SAMN04514119,GCA_022316345.1
404,"MRSN 14170,""MRSN14170"",""SRS1308342""",PDT000111517.2,SAMN04514120,GCA_022316325.1
405,"MRSN 14213,""MRSN14213"",""SRS1308341""",PDT000111518.2,SAMN04514121,GCA_022316305.1
407,"MRSN 20094,""MRSN20094"",""SRS1308339""",PDT000111520.2,SAMN04514123,GCA_022316285.1
408,"MRSN 20097,""MRSN20097"",""SRS1308371""",PDT000111521.2,SAMN04514124,GCA_022316195.1
